# Exercise 3

In [ ]:
import locale

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    locale.setlocale(locale.LC_ALL, 'fr_FR.utf8')
except locale.Error:
    pass

production_data_path = 'edunao-files/Donnees_France_2024_fr_cet.csv'
capacity_data_path = 'edunao-files/Capacites_France_2024.csv'

france_2024_data = pd.read_csv(production_data_path, sep=';', decimal=',', index_col=0)
france_2024_data.index = pd.to_datetime(france_2024_data.index, utc=True).tz_convert('Europe/Paris')
france_2024_data = france_2024_data.apply(pd.to_numeric, errors='coerce')

installed_capacity_data = pd.read_csv(capacity_data_path, sep=';', decimal=',', index_col=0)
installed_capacity_data.index = pd.to_datetime(installed_capacity_data.index, utc=True).tz_convert('Europe/Paris')
installed_capacity_data = installed_capacity_data.apply(pd.to_numeric, errors='coerce')

must_run_columns = ['Wind Onshore', 'Solar', 'Hydro Run-of-river and poundage']
gross_consumption_mw = france_2024_data['Load']
must_run_production_mw = france_2024_data[must_run_columns].fillna(0).sum(axis=1)
net_consumption_mw = gross_consumption_mw - must_run_production_mw
net_load_duration_mw = net_consumption_mw.sort_values(ascending=False).reset_index(drop=True)

plant_data = pd.DataFrame(
    {
        'Efficiency': [0.35, 0.40, 0.60, 0.35],
        'CO2 intensity (t/MWh_e)': [0.0, 1.0, 0.4, 0.6],
        'Capex (MEUR/MW_e)': [6.0, 1.5, 1.0, 0.7],
        'Fuel cost (EUR/MWh_th)': [5.0, 10.0, 30.0, 50.0],
    },
    index=['Nuclear', 'Coal', 'Gas CCGT', 'Oil OCGT'],
)

annuity_rate = 0.08
co2_price_eur_per_t = 80
hours_per_year = len(net_consumption_mw)

Month = mdates.MonthLocator(bymonthday=1)
MonthFmt = mdates.DateFormatter('%b')

## 3.1 Question

For each type of plant, determine the annual investment cost by considering an annuity equal to 8% of the total investment cost of the plant.

In [ ]:
plant_data['Annual investment cost (EUR/MW/year)'] = (
    plant_data['Capex (MEUR/MW_e)'] * 1_000_000 * annuity_rate
)

investment_results = plant_data[['Capex (MEUR/MW_e)', 'Annual investment cost (EUR/MW/year)']].copy()
investment_results['Annual investment cost (kEUR/MW/year)'] = (
    investment_results['Annual investment cost (EUR/MW/year)'] / 1000
)

display(investment_results.drop(columns='Annual investment cost (EUR/MW/year)').round(1))

## 3.2 Question

For each type of plant, determine the fuel cost required to produce 1 MWh of electricity.

In [ ]:
plant_data['Fuel cost (EUR/MWh_e)'] = (
    plant_data['Fuel cost (EUR/MWh_th)'] / plant_data['Efficiency']
)

fuel_results = plant_data[['Efficiency', 'Fuel cost (EUR/MWh_th)', 'Fuel cost (EUR/MWh_e)']]
display(fuel_results.round(2))

## 3.3 Question

For each type of plant, determine the CO2 emission cost associated with producing 1 MWh of electricity, using a CO2 price of 80 EUR/tCO2.

In [ ]:
plant_data['CO2 cost (EUR/MWh_e)'] = (
    plant_data['CO2 intensity (t/MWh_e)'] * co2_price_eur_per_t
)
plant_data['Variable cost (EUR/MWh_e)'] = (
    plant_data['Fuel cost (EUR/MWh_e)'] + plant_data['CO2 cost (EUR/MWh_e)']
)

emission_cost_results = plant_data[
    ['CO2 intensity (t/MWh_e)', 'CO2 cost (EUR/MWh_e)', 'Variable cost (EUR/MWh_e)']
]
display(emission_cost_results.round(2))

## 3.4 Question

For 1 MW of production capacity, calculate the total production cost as a function of the number of annual operating hours. Plot these curves on the same axes for each type of plant.

In [ ]:
operating_hours = np.arange(0, hours_per_year + 1)
screening_costs = pd.DataFrame(index=operating_hours)

for plant in plant_data.index:
    screening_costs[plant] = (
        plant_data.loc[plant, 'Annual investment cost (EUR/MW/year)']
        + plant_data.loc[plant, 'Variable cost (EUR/MWh_e)'] * operating_hours
    )

fig, ax = plt.subplots(figsize=(8, 4))
for plant in screening_costs.columns:
    ax.plot(screening_costs.index, screening_costs[plant] / 1000, label=plant)
plt.title('Annual Production Cost for 1 MW of Capacity')
plt.xlabel('Annual operating hours (h/year)')
plt.ylabel('Annual cost (kEUR/MW/year)')
ax.set_xlim(0, hours_per_year)
plt.grid(True)
plt.legend()
fig.tight_layout()
fig.savefig('figures/exercise_3-4.pdf')
plt.show()

display(plant_data[['Annual investment cost (EUR/MW/year)', 'Variable cost (EUR/MWh_e)']].round(2))

## 3.5 Question

Determine the operating-hour intervals during which each type of plant is the most competitive.

In [ ]:
cheapest_plant = screening_costs.idxmin(axis=1)

competitive_segments = []
segment_start = 0
current_plant = cheapest_plant.iloc[0]

for hour, plant in cheapest_plant.iloc[1:].items():
    if plant != current_plant:
        competitive_segments.append((segment_start, hour - 1, current_plant))
        segment_start = hour
        current_plant = plant

competitive_segments.append((segment_start, hours_per_year, current_plant))

competitive_intervals = pd.DataFrame(
    competitive_segments,
    columns=['From operating hour', 'To operating hour', 'Least-cost plant'],
)

display(competitive_intervals)

print('Comment:')
print('- Oil is least expensive only for rare peak operation because its fixed cost is low and its variable cost is high.')
print('- Gas CCGT is least expensive for intermediate operation.')
print('- Nuclear is least expensive for long annual operating durations because its variable cost is low despite high investment cost.')
print('- Coal is never on the least-cost envelope with the assumed CO2 price.')

## 3.6 Question

Using the load duration curve from the previous exercise, determine the production capacities of each plant type that minimize total production cost. Compare these values with the installed fleet in France.

In [ ]:
def load_at_duration(hours):
    if hours <= 0:
        return float(net_load_duration_mw.iloc[0])
    if hours >= len(net_load_duration_mw):
        return 0.0
    return float(net_load_duration_mw.iloc[int(np.ceil(hours))])

optimal_capacities_mw = pd.Series(0.0, index=plant_data.index)

for start_hour, end_hour, plant in competitive_segments:
    upper_load = load_at_duration(start_hour)
    lower_load = load_at_duration(end_hour + 1)
    optimal_capacities_mw.loc[plant] += max(upper_load - lower_load, 0.0)

installed_comparison_mw = pd.Series(
    {
        'Nuclear': installed_capacity_data.iloc[0]['Nuclear'],
        'Coal': installed_capacity_data.iloc[0]['Fossil Hard coal'],
        'Gas CCGT': installed_capacity_data.iloc[0]['Fossil Gas'],
        'Oil OCGT': installed_capacity_data.iloc[0]['Fossil Oil'],
    }
)

capacity_comparison = pd.DataFrame(
    {
        'Optimal simplified capacity (GW)': optimal_capacities_mw / 1000,
        'Installed French capacity (GW)': installed_comparison_mw / 1000,
    }
).round(2)

display(capacity_comparison)

print('Comment:')
print('- The simplified optimum is sized only to meet net demand after must-run wind, solar, and run-of-river hydro.')
print('- The resulting nuclear capacity is lower than installed French nuclear capacity because exports, reserves, maintenance margins, and operational constraints are ignored.')
print('- Gas and oil capacities are higher than the installed values because the simplified model has no imports, storage, demand response, or detailed hydro reservoir dispatch.')

## 3.7 Question

Plot the production curves of these generation sources over the year, then zoom in on one week. Compare them with the real production data and comment on the results.

In [ ]:
nuclear_capacity_mw = optimal_capacities_mw.loc['Nuclear']
gas_capacity_mw = optimal_capacities_mw.loc['Gas CCGT']
oil_capacity_mw = optimal_capacities_mw.loc['Oil OCGT']
modeled_generation_mw = pd.DataFrame(index=net_consumption_mw.index)
modeled_generation_mw['Nuclear'] = np.minimum(net_consumption_mw, nuclear_capacity_mw)
modeled_generation_mw['Gas CCGT'] = np.minimum(
    np.maximum(net_consumption_mw - nuclear_capacity_mw, 0),
    gas_capacity_mw,
)
modeled_generation_mw['Oil OCGT'] = np.minimum(
    np.maximum(net_consumption_mw - nuclear_capacity_mw - gas_capacity_mw, 0),
    oil_capacity_mw,
)
modeled_generation_mw['Coal'] = 0.0
real_generation_mw = pd.DataFrame(
    {
        'Nuclear': france_2024_data['Nuclear'],
        'Gas CCGT': france_2024_data['Fossil Gas'],
        'Oil OCGT': france_2024_data['Fossil Oil'],
        'Coal': france_2024_data['Fossil Hard coal'],
    },
    index=france_2024_data.index,
)
real_generation_labels = {
    'Nuclear': 'Real nuclear',
    'Gas CCGT': 'Real fossil gas',
    'Oil OCGT': 'Real fossil oil',
    'Coal': 'Real coal',
}
fig, ax = plt.subplots(figsize=(11, 4))
for plant, color in [('Nuclear', 'tab:blue'), ('Gas CCGT', 'tab:orange'), ('Oil OCGT', 'tab:red')]:
    ax.plot(modeled_generation_mw.index, modeled_generation_mw[plant] / 1000, linewidth=.6, color=color, label=f'Modeled {plant}')
    ax.plot(real_generation_mw.index, real_generation_mw[plant] / 1000, linewidth=.4, linestyle='--', color=color, alpha=.7, label=real_generation_labels[plant])
ax.plot(real_generation_mw.index, real_generation_mw['Coal'] / 1000, linewidth=.4, linestyle='--', color='tab:gray', alpha=.7, label='Real coal')
plt.title('Modeled and Real Dispatchable Production in France, 2024')
ax.set_xlim(net_consumption_mw.index[0], net_consumption_mw.index[-1])
ax.xaxis.set_major_locator(Month)
ax.xaxis.set_major_formatter(MonthFmt)
plt.xlabel('Month')
plt.ylabel('Power (GW)')
plt.grid(True)
plt.legend(ncol=2, fontsize=8)
fig.tight_layout()
fig.savefig('figures/exercise_3-7_annual.pdf')
plt.show()
week_start = pd.Timestamp('2024-01-08', tz='Europe/Paris')
week_end = pd.Timestamp('2024-01-15', tz='Europe/Paris')
week_slice = slice(week_start, week_end)
fig, ax = plt.subplots(figsize=(11, 4))
for plant, color in [('Nuclear', 'tab:blue'), ('Gas CCGT', 'tab:orange'), ('Oil OCGT', 'tab:red')]:
    ax.plot(modeled_generation_mw.loc[week_slice].index, modeled_generation_mw.loc[week_slice, plant] / 1000, linewidth=1.0, color=color, label=f'Modeled {plant}')
    ax.plot(real_generation_mw.loc[week_slice].index, real_generation_mw.loc[week_slice, plant] / 1000, linewidth=.8, linestyle='--', color=color, alpha=.7, label=real_generation_labels[plant])
ax.plot(real_generation_mw.loc[week_slice].index, real_generation_mw.loc[week_slice, 'Coal'] / 1000, linewidth=.8, linestyle='--', color='tab:gray', alpha=.7, label='Real coal')
plt.title('Modeled and Real Dispatchable Production, 8-15 January 2024')
plt.xlabel('Date')
plt.ylabel('Power (GW)')
plt.grid(True)
plt.legend(ncol=2, fontsize=8)
fig.tight_layout()
fig.savefig('figures/exercise_3-7_week.pdf')
plt.show()
modeled_energy_twh = modeled_generation_mw.sum() / 1_000_000
real_energy_twh = real_generation_mw.sum() / 1_000_000
dispatch_comparison = pd.DataFrame(
    {
        'Modeled annual energy (TWh)': modeled_energy_twh,
        'Real annual energy (TWh)': real_energy_twh,
    }
).round(2)
display(dispatch_comparison)
print('Comment:')
print('- The modeled dispatch follows a strict merit-order stack and therefore uses nuclear as a flat base layer, then gas, then oil during peak periods.')
print('- Real production is smoother and constrained by maintenance, ramping, reserves, hydro reservoir management, imports, exports, and market operation.')
print('- The simplified model should be interpreted as an economic sizing exercise, not as a realistic unit-commitment simulation.')


## 3.8 Question

Calculate the investment, fuel, and CO2 costs of each type of plant required to satisfy the net demand.

In [ ]:
modeled_energy_mwh = modeled_generation_mw.sum()

cost_results = pd.DataFrame(index=plant_data.index)
cost_results['Capacity (GW)'] = optimal_capacities_mw / 1000
cost_results['Annual energy (TWh)'] = modeled_energy_mwh / 1_000_000
cost_results['Investment cost (bn EUR/year)'] = (
    optimal_capacities_mw * plant_data['Annual investment cost (EUR/MW/year)'] / 1_000_000_000
)
cost_results['Fuel cost (bn EUR/year)'] = (
    modeled_energy_mwh * plant_data['Fuel cost (EUR/MWh_e)'] / 1_000_000_000
)
cost_results['CO2 cost (bn EUR/year)'] = (
    modeled_energy_mwh * plant_data['CO2 cost (EUR/MWh_e)'] / 1_000_000_000
)
cost_results['Total cost (bn EUR/year)'] = cost_results[
    ['Investment cost (bn EUR/year)', 'Fuel cost (bn EUR/year)', 'CO2 cost (bn EUR/year)']
].sum(axis=1)

display(cost_results.fillna(0).round(3))

total_cost = cost_results['Total cost (bn EUR/year)'].sum()
total_net_energy = net_consumption_mw.sum() / 1_000_000
average_cost_eur_per_mwh = total_cost * 1_000_000_000 / (total_net_energy * 1_000_000)

print(f'Total modeled annual cost: {total_cost:.2f} bn EUR/year')
print(f'Total net demand served: {total_net_energy:.1f} TWh')
print(f'Average modeled cost: {average_cost_eur_per_mwh:.1f} EUR/MWh')